In [0]:
from pyspark.sql import functions as F

# Parameterize the month to load — lets us re-run this notebook for each new month
dbutils.widgets.text("year_month", "2024-01", "Year-Month to ingest (YYYY-MM)")
year_month = dbutils.widgets.get("year_month")

CATALOG = "workspace"
SCHEMA = "default"
VOLUME_PATH = "/Volumes/workspace/default/raw_files"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

In [0]:
raw_file = f"{VOLUME_PATH}/yellow_tripdata_{year_month}.parquet"

trips_df = (
    spark.read.parquet(raw_file)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit(raw_file))
    .withColumn("_year_month", F.lit(year_month))
)

print(f"Rows loaded for {year_month}: {trips_df.count():,}")
display(trips_df.limit(5))

In [0]:
spark.sql(f"""
    DELETE FROM {CATALOG}.{SCHEMA}.trips_raw
    WHERE _year_month = '{year_month}'
""")
print(f"Cleared any existing rows for {year_month} before reload")

In [0]:
(
    trips_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.trips_raw")
)

print("Trips written to bronze table.")

In [0]:
zones_df = (
    spark.read.option("header", "true").csv(f"{VOLUME_PATH}/taxi_zone_lookup.csv")
    .withColumn("_ingested_at", F.current_timestamp())
)

(
    zones_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.zone_lookup_raw")
)

print("Zone lookup written to bronze table.")
display(zones_df.limit(5))

In [0]:
display(
    spark.table(f"{CATALOG}.{SCHEMA}.trips_raw")
    .groupBy("_year_month")
    .count()
    .orderBy("_year_month")
)